#

In [7]:
# ===================================================================
# Import all packages and declare custom functions

# General utilities:
import os 
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Stats
import pymc as pm
import arviz as az
import bambi as bmb
from scipy.special import logit, expit
from scipy.stats import pearsonr

# Custom packages:
from stabst.MarkovDecisionProcess import MDP
from stabst.TaskConfig import LimitedEnergyTask
from stabst.utils import avg_reduce_mdp, abstract2ground_value

A4_inches = [8.27-0.5,11.69-0.5]
rng = np.random.default_rng(615)

def make_grid(dv_min, dv_max, pref_min, pref_max):
    """
    Makes a 2D grid reflecting the limits of our modelled variables (preferences and DV)
    Parameters
    ----------
    dv_min : _type_
        _description_
    dv_max : _type_
        _description_
    pref_min : _type_
        _description_
    pref_max : _type_
        _description_

    Returns
    -------
    _type_
        _description_
    """
    x1_grid = np.linspace(start=dv_min, stop=dv_max, num=300)
    x2_grid = np.linspace(start=pref_min, stop=pref_max, num=300)
    x1_mesh, x2_mesh = np.meshgrid(x1_grid, x2_grid)
    x_grid = np.stack(arrays=[x1_mesh.flatten(), x2_mesh.flatten()], axis=1)
    return x1_grid, x2_grid, x_grid

# Define custom functions:
# Choice behaviour model with preferences and DV:
def preference_model(
        y: np.array,
        decision_values: np.array,
        pref_regressors: pd.DataFrame,
        subject_index: np.array,
        subject_labels: np.array      
):
    '''
    Parameters
    ----------
    y : np.array [N samples, ]
        Observed binary data (1, 0...)
    decision_values : np.array [N samples, ]
        DV to regress onto the observed data
    pref_regressors : np.array [N samples, M regressors]
        Regressor to fit participants preference for. We can have M regressors
    subject_index : np.array  [N samples, ]
        Index of the subject associated with each observation
    subject_labels : np.array  [N subjects, ]
        Single identifier of each subject
    coords : dict  
        "subject": subj_labels, 
        "coef": ["intercept", "slope"],
        The subject maps the data to each subject, the coef are for the coefficients
    b_prior_mean : Optional[float], optional
        Prior mean of each beta parameters, by default 0
    b_prior_sigma : Optional[float], optional
        Prior variance of the population level distribution of the beta, by default 2`
    s_prior_sigma : Optional[float], optional
        Prior between subjects variance, by default 2
    n_drawss : Optional[int], optional
        Number of draws for the posterior, by default 1000
    n_tuning_draws : Optional[int], optional
        Number of tuning draws, by default 1000
    Returns
    -------
    tuple[pm.Model, arviz.InferenceData]
        pm.model : pymc model object
        idata : arviz inference data
    '''
    # Get dimensions:
    n_obs = y.shape[0]
    n_groups = subject_labels.shape[0]
    n_pref = pref_regressors.shape[1]

    # Create intercept:
    intercept = np.ones(n_obs)

    # Set coordinates:
    coords = {
        "subject": subject_labels,
        "coef_intercept": ["B_intercept"],
        "coef_planning": ["B_plan"],
        "coef_pref": ["B_" + col for col in pref_regressors.columns],
        "coef_interaction": ["slope"],
    }


    # Model:
    with pm.Model(coords=coords) as planning_preferences_interaction_model:
        # Data:
        y_obs = pm.Data("y_obs", y)
        intercept = pm.Data("intercept", intercept)
        planning = pm.Data("planning", decision_values)
        preferences = pm.Data("preferences", pref_regressors)
        subj_idx = pm.Data("subj_idx", subject_index.astype("int32"))

        # Hyperpriors:
        # Intercept term
        beta_intercept = pm.Normal("beta_intercept", mu=0, sigma=2, dims="coef_intercept")
        sigma_intercept = pm.HalfNormal("sigma_intercept", sigma=2, dims="coef_intercept")
        # Planning term:
        beta_planning = pm.Normal("beta_planning", mu=0, sigma=2, dims="coef_planning")
        sigma_planning = pm.HalfNormal("sigma_planning", sigma=2, dims="coef_planning")
        # Preference terms:
        beta_pref = pm.Normal("beta_pref", mu=0, sigma=2, dims="coef_pref")
        sigma_pref = pm.HalfNormal("sigma_pref", sigma=2, dims="coef_pref")
        # Interaction term:
        beta_interaction = pm.Normal("beta_interaction", mu=0, sigma=2, dims="coef_interaction")
        sigma_interaction = pm.HalfNormal("sigma_interaction", sigma=2, dims="coef_interaction")

        # Offset parameters:
        z_intercept = pm.Normal("z_intercept", 0, 1, dims=("subject", "coef_intercept"))
        z_planning = pm.Normal("z_planning", 0, 1, dims=("subject", "coef_planning"))
        z_biases = pm.Normal("z_biases", 0, 1, dims=("subject", "coef_pref"))
        z_interaction = pm.Normal("z_interaction", 0, 1, dims=("subject", "coef_interaction"))

        # Centered parameters:
        beta_intercept_sub = pm.Deterministic("beta_intercept_sub", beta_intercept + z_intercept * sigma_intercept, 
                                              dims=("subject", "coef_intercept"))
        beta_planning_sub = pm.Deterministic("beta_planning_sub", beta_planning + z_planning * sigma_planning, 
                                             dims=("subject", "coef_planning"))
        beta_pref_sub = pm.Deterministic("beta_pref_sub", beta_pref + z_biases * sigma_pref, 
                                         dims=("subject", "coef_pref"))
        beta_interaction_sub = pm.Deterministic("beta_interaction_sub", beta_interaction + z_interaction * sigma_interaction, 
                                                dims=("subject", "coef_interaction"))
        
        # Estimate the score of the bias (i.e. weighted sum of each of the biases regressors):
        preference = pm.Deterministic('preference', (beta_pref_sub[subj_idx] * preferences).sum(axis=-1))
        
        # Convert the bias back onto probability space:
        pi_prior = pm.Deterministic("pi_prior", pm.math.sigmoid(preference))

        # Compute the normalized entropy (defined between 0 and 1):
        max_entropy = -0.5 * np.log(0.5) - (1-0.5) * np.log(1 - 0.5)
        entropy = pm.Deterministic("entropy", (-pi_prior * pm.math.log(pi_prior) - (1-pi_prior) * pm.math.log(1 - pi_prior)) / max_entropy)
        
        # Eta parameter is the weighted sum of the intercept, the bias, the planning values and 
        # the interaction between the entropy of the bias and the planning
        eta = (
            beta_intercept_sub[subj_idx, 0] * intercept
            + preference
            + beta_planning_sub[subj_idx, 0] * planning
            + beta_interaction_sub[subj_idx, 0] * (entropy * planning)
        )
        
        # Expected values:
        p = pm.Deterministic("p", pm.math.sigmoid(eta))

        # Likelihood 
        pm.Bernoulli("y", p=p, observed=y_obs)

        # Sampling:
        idata = pm.sample(
            draws=n_samples,
            tune=n_warmup,
            chains=n_chains,
            target_accept=0.85,
            idata_kwargs={"log_likelihood": True},
            random_seed=rng
        )
        # Sample posterior predictive for later model checking:
        idata.extend(pm.sample_posterior_predictive(idata, 
                                                    var_names=["p", "y"],
                                                    random_seed=rng),
                                                    )
    return idata

def linear_decay_mvavg(arr, window_size, init_val = 0.5):
    '''
    Compute the moving average of a 1D array with a specified window size and 
    linear decay
    '''
    # Pad the array with initial value
    arr = np.pad(arr.astype(float), (window_size - 1, 0), constant_values=init_val)
    # Generate kernel:
    kern = np.arange(window_size)/np.sum(np.arange(window_size))
    return np.convolve(arr, kern, mode='valid')

def predict_responses(dv, pref, idata):
    """_summary_

    Parameters
    ----------
    dv : _type_
        _description_
    pref : _type_
        _description_
    beta_plan : _type_
        _description_
    beta_interaction : _type_
        _description_
    """
    # Compute the preferences entropy:
    p_pref = expit(pref)
    entropy = -p_pref * np.log(p_pref) - (1-p_pref) * np.log(1 - p_pref)

    # Stack all posterior samples (chains and draws)
    b_plan = idata.posterior["beta_planning"].stack(sample=("chain", "draw")).values.squeeze()  # shape (n_samples,)
    b_inter = idata.posterior["beta_interaction"].stack(sample=("chain", "draw")).values.squeeze()  # shape (n_samples,)

    # Compute predictions for each sample and input
    eta = b_plan[:, None] * dv + pref + b_inter[:, None] * dv * entropy
    pred = expit(eta)  # shape (n_samples, n_points)

    # Compute mean and 95% credible interval
    mean_pred = np.mean(pred, axis=0)
    lower_95 = np.percentile(pred, 2.5, axis=0)
    upper_95 = np.percentile(pred, 97.5, axis=0)

    return mean_pred, lower_95, upper_95

def truncate_colormap(cmap, min_val=0.0, max_val=1.0, n=256):
    """Slice a colormap to the sub-range [min_val, max_val]."""
    colors = cmap(np.linspace(min_val, max_val, n))
    return mcolors.LinearSegmentedColormap.from_list(
        f"{cmap.name}_trunc({min_val:.2f},{max_val:.2f})", colors
    )

# Set random seed for reproducibility:
np.random.seed(42)
# warnings.filterwarnings("ignore")

n_samples = 1000
n_warmup = 3000
n_chains = 4

In [8]:
# ===================================================================
# Download and prepare data:
if not os.path.exists('./data/raw_data/all_participants_data.csv'):
    if not os.path.exists('./data/raw_data'):
        os.makedirs('./data/raw_data')
    url = 'https://raw.githubusercontent.com/fmott/context_dependent_planning/4d239b721749adabb8fe8f1d8ac2d1ecdeba17cf/data/behaviour/data_all_participants_20220215120148.csv'
    os(f'wget {url} -O ./data/raw_data/all_participants_data.csv')
if not os.path.exists('./data/raw_data/all_participants_age_gender.csv'):
    if not os.path.exists('./data/raw_data'):
        os.makedirs('./data/raw_data')
    url = 'https://raw.githubusercontent.com/fmott/context_dependent_planning/4d239b721749adabb8fe8f1d8ac2d1ecdeba17cf/data/behaviour/age_gender.csv'
    os(f'wget {url} -O ./data/raw_data/all_participants_age_gender.csv')
# Load the data:
beh_data = pd.read_csv('./data/raw_data/all_participants_data.csv')
demographic_data = pd.read_csv('./data/raw_data/all_participants_age_gender.csv', sep=";")

# Extract demographic information:
n_participants = demographic_data.shape[0]
n_female = demographic_data['gender (m = 1, f = 2)'].value_counts()[2]
mean_age = demographic_data['age'].mean()
std_age = demographic_data['age'].std()

# Calcuate the number of trials rejected & RT in each subject:
n_rej = []
rt = []
n_score = []
for sub in np.unique(beh_data['vpn']):
    # Extract subject's data:
    sub_data = beh_data[beh_data['vpn'] == sub]
    n_na = sub_data.shape[0] - sub_data.dropna().shape[0]
    n_rej.append(n_na)
    rt.append(sub_data.dropna()['reaction_time'].mean())
    n_score.append(sub_data.dropna()['points'].to_numpy()[-1])
# Get average and standard deviation
mean_timeout = np.mean(n_rej)
sd_timeout = np.std(n_rej)
mean_rt = np.mean(rt)
sd_rt = np.std(rt)
mean_points = np.mean(n_score)
sd_points = np.std(n_score)

# ===================================================================
# Data preprocessing:
# Remove nans:
beh_data = beh_data.dropna()
# Remove timeout:
beh_data = beh_data[beh_data["timeout"] == 0]
# Flip responses: 1 = accept:
beh_data["response"] = (beh_data["response"] == 0).astype(int)
# Make trial 1 based
beh_data["trial"] = beh_data["trial"] + 1
# Generate future cost based on the transitions:
transitions_costs = {
    0: [1, 1],
    1: [2, 1],
    2: [1, 2],
    3: [2, 2]
}
beh_data["fc"] = [transitions_costs[row["transition"]][1] for _, row in beh_data.iterrows()]

In [9]:
# ===================================================================
# Set up task MDP and calcuate DV
# Create the task and its parameters (transition probability, reward...):
task = LimitedEnergyTask(O=[1, 2, 3, 4], p_offer=[1/4] * 4)
task.build()

# Create full MDP and compute solution for later reference:
gamma = 1
task_mdp = MDP(task.states, task.tp, task.r, gamma, s2i=task.s2i)
V_full, Q_full = task_mdp.backward_induction()

# Add DV to the data frame:
dv = Q_full[:, 1] - Q_full[:, 0]
# Loop through each trial to set DV:
dv_trials = []
for trial_i, trial in beh_data.iterrows():
    e, o, cc, t = trial.energy, trial.reward, trial.energy_cost, trial.trial
    fc = transitions_costs[trial.transition][1]
    dv_trials.append(dv[task.s2i[(e, o, cc, fc, t)]])
beh_data['dv'] = dv_trials
# Compute offer specific decision value regressors:
beh_data['dv_23'] = beh_data['dv'].to_numpy() * (beh_data['is_2'].to_numpy() + beh_data['is_3'].to_numpy())
beh_data['dv_14'] = beh_data['dv'].to_numpy() * (beh_data['is_1'].to_numpy() + beh_data['is_4'].to_numpy())

In [10]:
# ===================================================================
# Prepare regressors for all models:
# Compute categorical regressors that should be somewhat similar to our priors:
# Categorical offer regressor for high and low offer
beh_data['is_12'] = beh_data['is_1'].to_numpy() + beh_data['is_2'].to_numpy()
beh_data['is_34'] = beh_data['is_3'].to_numpy() + beh_data['is_4'].to_numpy()
# Continuous regressor for high and low offer:
beh_data['high_vs_low'] = beh_data['is_34'] - beh_data['is_12']

# Categorical costs regressor
beh_data['is_lc'] = (beh_data['energy_cost'] == 1).astype(int).to_numpy()
beh_data['is_hc'] = (beh_data['energy_cost'] == 2).astype(int).to_numpy()
# Categorical future costs regressor
beh_data['is_lfc'] = (beh_data['fc'] == 1).astype(int).to_numpy()
beh_data['is_hfc'] = (beh_data['fc'] == 2).astype(int).to_numpy()

# Categorical transition regressor
beh_data['is_trans1'] = (beh_data['transition'] == 0).to_numpy()
beh_data['is_trans2'] = (beh_data['transition'] == 1).to_numpy()
beh_data['is_trans3'] = (beh_data['transition'] == 2).to_numpy()
beh_data['is_trans4'] = (beh_data['transition'] == 3).to_numpy()

# Categorical energy regressor:
beh_data['e_is_0'] = (beh_data['energy'] == 0).to_numpy()
beh_data['e_is_1'] = (beh_data['energy'] == 1).to_numpy()
beh_data['e_is_2'] = (beh_data['energy'] == 2).to_numpy()
beh_data['e_is_3'] = (beh_data['energy'] == 3).to_numpy()
beh_data['e_is_4'] = (beh_data['energy'] == 4).to_numpy()
beh_data['e_is_5'] = (beh_data['energy'] == 5).to_numpy()
beh_data['e_is_6'] = (beh_data['energy'] == 6).to_numpy()

# Random effects
subj_idx_raw, subj_labels = pd.factorize(beh_data["vpn"])

In [11]:
traces = {}

# Fit alternative models:

# ==========================================================================================
# Original pref model
idata = az.from_netcdf("./data/bids/limited_energy/derivatives/models/preferences_model_trace.nc")
traces['Preference_ori'] = idata

# ==========================================================================================
# Original context model
idata = az.from_netcdf("./data/bids/limited_energy/derivatives/models/hybrid_model_trace.nc")
traces['Context'] = idata

# ==========================================================================================
# Original frequency model
idata = az.from_netcdf("./data/bids/limited_energy/derivatives/models/action_prior_model_trace.nc")
traces['Frequency prior'] = idata

# ==========================================================================================
# Original planning model:
idata = az.from_netcdf("./data/bids/limited_energy/derivatives/models/planning_model_trace.nc")
traces['Planning'] = idata

In [12]:
loo_ori = az.loo(traces["Preference_ori"], pointwise=True)

# Plot the pareto K distributions:
loo_dict = {}
for mdl in traces.keys():
    # compute pointwise LOO
    loo = az.loo(traces[mdl], pointwise=True)
    loo_vals = loo.loo_i[np.where(loo_ori.pareto_k <= 0.7)[0]]
    loo_dict[mdl] = np.sum(loo_vals)
print(loo_dict)

/home/alex-lepauvre/miniforge3/envs/pymc_env/lib/python3.14/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
/home/alex-lepauvre/miniforge3/envs/pymc_env/lib/python3.14/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
/home/alex-lepauvr

{'Preference_ori': <xarray.DataArray 'loo_i' ()> Size: 8B
array(-1455.35728787), 'Context': <xarray.DataArray 'loo_i' ()> Size: 8B
array(-1621.54578996), 'Frequency prior': <xarray.DataArray 'loo_i' ()> Size: 8B
array(-2026.28668798), 'Planning': <xarray.DataArray 'loo_i' ()> Size: 8B
array(-2032.31808126)}

In [13]:
# ==========================================================================================
# Upsampled pref model
preference_columns = [
    'is_1', 'is_2', 'is_3', 'is_4', 
    'is_lc', 'is_hc', 
    'is_lfc', 'is_hfc', 
    'e_is_0', 'e_is_1', 'e_is_2', 'e_is_3', 'e_is_4', 'e_is_5', 'e_is_6'
]
idata = preference_model(beh_data['response'],  # Subjects responses
                            np.squeeze(beh_data[['dv']].to_numpy()),  # Optimal decision values
                            beh_data[preference_columns],  # Preferences regressors
                            subj_idx_raw, subj_labels)

# Add the idata to the rest:
traces['Preference_upsampled'] = idata

# ==========================================================================================
# Lighter
preference_columns = [
    'is_1', 'is_2', 'is_3', 'is_4', 
    'is_lc',
    'is_lfc',
    'e_is_0', 'e_is_1', 'e_is_5', 'e_is_6'
]
idata = preference_model(beh_data['response'],  # Subjects responses
                            np.squeeze(beh_data[['dv']].to_numpy()),  # Optimal decision values
                            beh_data[preference_columns],  # Preferences regressors
                            subj_idx_raw, subj_labels)

# Add the idata to the rest:
traces['Preference_light'] = idata

# ==========================================================================================
# Preferences no interactions:
pref_no_interactions = bmb.Model(
    "response ~ dv + is_1 + is_2 + is_3 + is_4 + is_lc + is_hc + is_lfc + is_hfc + e_is_0 + e_is_1 + e_is_2 + e_is_3 + e_is_4 + e_is_5 + e_is_6 + (dv+ is_1 + is_2 + is_3 + is_4 + is_lc + is_hc + is_lfc + is_hfc + e_is_0 + e_is_1 + e_is_2 + e_is_3 + e_is_4 + e_is_5 + e_is_6|vpn)",
        beh_data, 
        family="bernoulli"
    )
# Add the idata to the rest:
traces['pref_no_interactions'] = pref_no_interactions.fit(
    draws=1000, 
    tune=1000, 
    chains=n_chains, 
    target_accept=0.85, 
    idata_kwargs={"log_likelihood": True},
    random_seed=rng
)

# Plot the model comparison:
model_comparison = az.compare(traces)
az.plot_compare(model_comparison);

# Plot the pareto K distributions:
for mdl in traces.keys():
    # compute pointwise LOO
    loo = az.loo(traces[mdl], pointwise=True)
    az.plot_khat(loo.pareto_k);

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_intercept, sigma_intercept, beta_planning, sigma_planning, beta_pref, sigma_pref, beta_interaction, sigma_interaction, z_intercept, z_planning, z_biases, z_interaction]

In [ ]:
loo_upsampled = az.loo(traces["Preference_upsampled"], pointwise=True)
loo_ori = az.loo(traces["Preference_ori"], pointwise=True)

# Check the ones that are larger than 1:
ind_excess = np.where(loo_upsampled.pareto_k > 0.7)[0]
ind_excess_ori = np.where(loo_ori.pareto_k >  0.7)[0]

print(len(ind_excess))
print(len(ind_excess_ori))